# OR Intelligence · 02 · Detection model (Member B)

Trains, evaluates and saves the instrument detector. Needs the dataset from **01_Data** (Member A).

| Step | What happens | WBS |
|---|---|---|
| 1 | Settings (the only cell you edit) | — |
| 2 | GPU check, install, connect Google Drive | — |
| 3 | Load Member A's dataset from Drive | — |
| 4 | Train the baseline model | 3.1.1 |
| 5 | Optional: one experiment | 3.1.2 |
| 6 | Pick the best model (validation split only) | 3.1.2 |
| 7 | Test-set evaluation (**runs once**) | 3.2.1 |
| 8 | Save the model to Drive | 3.3.2 |
| 9 | Run the model on a video → `detections.jsonl` | 3.3.1 |

**How to use:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Training saves straight to Drive with a checkpoint every epoch, so a disconnect does not lose work.

> Never commit `.pt` weights to GitHub. They live in `MyDrive/or-intelligence/weights/`.

## 1 · Settings — the only cell you need to edit

In [6]:
DRIVE_DIR   = "/content/drive/MyDrive/or-intelligence"
DATASET_ZIP = f"{DRIVE_DIR}/data/yolo_dataset.zip"   # produced by 01_Data.ipynb (Member A)

MODEL    = "yolov8s.pt"   # COCO-pretrained start point (yolov8n.pt = faster, yolov8m.pt = stronger)
EPOCHS   = 50
IMGSZ    = 640
BATCH    = 16
PATIENCE = 10
SEED     = 42
CONF     = 0.25           # confidence threshold for the test report and video inference

## 2 · GPU, install, Google Drive

In [7]:
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> T4 GPU"
!pip -q install ultralytics

from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)


import os, json, random, re, shutil, zipfile, glob, hashlib, datetime
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

for sub in ["data", "runs", "weights", "results"]:
    os.makedirs(f"{DRIVE_DIR}/{sub}", exist_ok=True)
WORK = Path("/content/work")      # fast local disk; Drive keeps the results
WORK.mkdir(exist_ok=True)
print("Drive folder ready:", DRIVE_DIR)

GPU 0: NVIDIA L4 (UUID: GPU-d39a769f-16ec-34a4-f0dc-697635b6ad13)
Mounted at /content/drive
Drive folder ready: /content/drive/MyDrive/or-intelligence


## 3 · Load the dataset from Drive

In [8]:
DS = WORK / "yolo"
if not DS.exists():
    assert os.path.exists(DATASET_ZIP), "Run 01_Data.ipynb first (Member A), or ask for yolo_dataset.zip"
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall(DS)
yaml_text = (DS / "dataset.yaml").read_text()
(DS / "dataset.yaml").write_text(re.sub(r"^path:.*$", lambda _: f"path: {DS}", yaml_text, flags=re.M))
CLASSES = [line.split(":", 1)[1].strip() for line in yaml_text.splitlines() if re.match(r"\s+\d+:", line)]
print("classes:", CLASSES)
print({k: len(list((DS / "images" / k).iterdir())) for k in ["train", "val", "test"]}, "images per split")

classes: ['grasper', 'bipolar', 'hook', 'scissors', 'clipper', 'irrigator']
{'train': 26830, 'val': 4284, 'test': 10819} images per split


## 4 · Train the baseline (WBS 3.1.1)

After it finishes, write the val mAP@0.5 into `docs/model/experiments.md`. If Colab disconnects, re-run steps 1–3 and then the **resume** cell.

In [9]:
from ultralytics import YOLO
import pandas as pd

def train(run_name, **overrides):
    start_from = overrides.pop("model", MODEL)
    args = dict(data=str(DS / "dataset.yaml"), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE,
                seed=SEED, deterministic=True, project=f"{DRIVE_DIR}/runs", name=run_name, exist_ok=True,
                save_period=1)
    args.update(overrides)
    YOLO(start_from).train(**args)
    return f"{DRIVE_DIR}/runs/{run_name}/weights/best.pt"

baseline_weights = train("baseline")
print("baseline weights:", baseline_weights)

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/work/yolo/dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline, nbs=64, nms=None, opset=None, optimize=False, o

In [15]:
# RESUME after a disconnect (only if training stopped early). Uncomment and run:
YOLO(f"{DRIVE_DIR}/runs/baseline/weights/last.pt").train(resume=True)

WARNING ⚠️ model '/content/drive/MyDrive/or-intelligence/runs/baseline/weights/last.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0, 16, 17, 20, 25, 58])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7abc3a2196a0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

## 5 · Optional: one experiment (WBS 3.1.2)

Change **one** thing, chosen from Member A's EDA (for example a larger image size if tools are small). Skip it if short on time.

In [16]:
RUN_EXPERIMENT = False      # set True to run it
if RUN_EXPERIMENT:
    exp_weights = train("exp01_imgsz832", imgsz=832)

## 6 · Pick the best model — validation split only (WBS 3.1.2)

The test split is **not** used for choosing.

In [12]:
def val_scores(weights):
    m = YOLO(weights).val(data=str(DS / "dataset.yaml"), split="val", imgsz=IMGSZ, verbose=False)
    return {"map50": float(m.box.map50), "map50_95": float(m.box.map),
            "per_class_ap50": {CLASSES[i]: float(m.box.ap50[j]) for j, i in enumerate(m.box.ap_class_index)}}

candidates = {name: f"{DRIVE_DIR}/runs/{name}/weights/best.pt" for name in ["baseline", "exp01_imgsz832"]
              if os.path.exists(f"{DRIVE_DIR}/runs/{name}/weights/best.pt")}
scores = {name: val_scores(w) for name, w in candidates.items()}
display(pd.DataFrame({n: {"val mAP@0.5": s["map50"], "val mAP@0.5:0.95": s["map50_95"]} for n, s in scores.items()}).T)
chosen = max(scores, key=lambda n: scores[n]["map50"])
BEST = candidates[chosen]
json.dump({"chosen": chosen, "scores": scores}, open(f"{DRIVE_DIR}/results/model_selection.json", "w"), indent=2)
print("chosen model:", chosen)

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2124.1±890.2 MB/s, size: 92.5 KB)
val: Scanning /content/work/yolo/labels/val.cache... 4284 images, 542 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4284/4284 1.4Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 268/268 11.3it/s 23.8s
                   all       4284       6380      0.931      0.803      0.868      0.773
Speed: 0.4ms preprocess, 1.9ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /content/runs/detect/val


,val mAP@0.5,val mAP@0.5:0.95
baseline,0.867736,0.772508


chosen model: baseline


## 7 · Test-set evaluation — runs ONCE (WBS 3.2.1)

mAP@0.5 and precision/recall per instrument on surgeries the model never saw. Copy the result into `docs/model/test_metrics.json` and `docs/model/EVALUATION.md`, and tell Member C the `CONF` value used.

In [13]:
TEST_FILE = f"{DRIVE_DIR}/results/test_metrics.json"
if os.path.exists(TEST_FILE):
    print("Test split already evaluated once - showing the saved result instead of re-running.")
    test = json.load(open(TEST_FILE))
else:
    m = YOLO(BEST).val(data=str(DS / "dataset.yaml"), split="test", imgsz=IMGSZ, conf=CONF, plots=True,
                       project=f"{DRIVE_DIR}/runs", name="test_eval", exist_ok=True)
    per_class = {}
    for j, i in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(j)
        per_class[CLASSES[i]] = {"precision": float(p), "recall": float(r), "ap50": float(ap50)}
    test = {"model": chosen, "weights_sha256": hashlib.sha256(open(BEST, "rb").read()).hexdigest(),
            "conf": CONF, "map50": float(m.box.map50), "map50_95": float(m.box.map), "per_class": per_class,
            "evaluated_at": datetime.datetime.now().isoformat(timespec="seconds")}
    json.dump(test, open(TEST_FILE, "w"), indent=2)
display(pd.DataFrame(test["per_class"]).T.round(3))
print(f"TEST mAP@0.5 = {test['map50']:.3f}   mAP@0.5:0.95 = {test['map50_95']:.3f}")

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2050.9±684.0 MB/s, size: 100.2 KB)
val: Scanning /content/work/yolo/labels/test... 10819 images, 1830 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 10819/10819 1.5Kit/s 7.3s
val: New cache created: /content/work/yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 677/677 12.0it/s 56.3s
                   all      10819      15659      0.927      0.857      0.881      0.786
               grasper       6728       8701      0.923      0.858      0.885      0.743
               bipolar        526        526      0.926      0.897      0.918      0.779
                  hook       4847       4848       0.99      0.973      0.983      0.886
              scissors        149        149      0.787      0.71

,precision,recall,ap50
grasper,0.923,0.858,0.885
bipolar,0.926,0.897,0.918
hook,0.990,0.973,0.983
scissors,0.787,0.711,0.711
clipper,0.965,0.855,0.908
irrigator,0.974,0.849,0.881


TEST mAP@0.5 = 0.881   mAP@0.5:0.95 = 0.786


## 8 · Save the final model (WBS 3.3.2)

`best.pt` (+ `best.onnx`, + sha256) go to `MyDrive/or-intelligence/weights/`. Share that folder with the team, or attach `best.pt` to a GitHub Release `model-v1`. Then write `docs/model/MODEL_CARD.md`.

In [17]:
final_pt = f"{DRIVE_DIR}/weights/best.pt"
shutil.copy(BEST, final_pt)
onnx_path = YOLO(final_pt).export(format="onnx", imgsz=IMGSZ)

dest_onnx = f"{DRIVE_DIR}/weights/best.onnx"
if os.path.abspath(onnx_path) != os.path.abspath(dest_onnx):
    shutil.copy(onnx_path, dest_onnx)

sha = hashlib.sha256(open(final_pt, "rb").read()).hexdigest()
open(f"{DRIVE_DIR}/weights/best.pt.sha256", "w").write(sha)
print("saved:", final_pt, "\nsha256:", sha)
# from google.colab import files; files.download(final_pt)     # optional: download to this computer

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/or-intelligence/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (21.5 MB)

ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming with onnxslim 0.1.99...
ONNX: export success ✅ 1.4s, saved as '/content/drive/MyDrive/or-intelligence/weights/best.onnx' (42.7 MB)

Export complete (1.7s)
Results saved to /content/drive/MyDrive/or-intelligence/weights/best.onnx
Predict:         yolo predict task=detect model=/content/drive/MyDrive/or-intelligence/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/drive/MyDrive/or-intelligence/weights/best.onnx imgsz=640 data=/content/work/yolo/dataset.yaml 
Visualize:       https://netron.app
saved: /content/drive/MyDrive/or-intelligence/weights/best.pt 
sha256:

## 9 · Run the model on a video → `detections.jsonl` (WBS 3.3.1)

This is the same function as `src/inference/detect_video.py` in the repo (which the app uses). Output follows contract C4 (`contracts/detections.schema.json`). Upload a video to Drive and set `VIDEO`.

In [18]:
import cv2

def detect_video(video_path, out_path, sample_fps=5.0, conf=CONF, weights=final_pt, on_progress=None):
    model = YOLO(weights)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    step = max(1, round(fps / sample_fps))
    vid, written, idx = Path(video_path).stem, 0, 0
    with open(out_path, "w") as f:
        while cap.grab():
            if idx % step == 0:
                ok, frame = cap.retrieve()
                if not ok:
                    break
                r = model.predict(frame, conf=conf, imgsz=IMGSZ, verbose=False)[0]
                dets = [{"cls": int(c), "name": CLASSES[int(c)], "conf": round(float(s), 4),
                         "box": [round(min(max(float(v), 0.0), 1.0), 4) for v in b]}
                        for c, s, b in zip(r.boxes.cls, r.boxes.conf, r.boxes.xyxyn)]
                f.write(json.dumps({"video_id": vid, "frame": idx, "t": round(idx / fps, 3), "w": W, "h": H, "dets": dets}) + "\n")
                written += 1
                if on_progress and total:
                    on_progress(min(idx / total, 1.0))
            idx += 1
    cap.release()
    if on_progress:
        on_progress(1.0)
    return {"frames": written, "seconds": (total or idx) / fps}

VIDEO = f"{DRIVE_DIR}/data/sample_video.mp4"     # upload any test video here
if os.path.exists(VIDEO):
    out = f"{DRIVE_DIR}/results/{Path(VIDEO).stem}_detections.jsonl"
    print(detect_video(VIDEO, out), "->", out)
else:
    print("No video at", VIDEO, "- skip this step or upload one.")

No video at /content/drive/MyDrive/or-intelligence/data/sample_video.mp4 - skip this step or upload one.


## Done — hand-off

| File in Drive | Who needs it | Repo location |
|---|---|---|
| `weights/best.pt` (+ `.onnx`, `.sha256`) | Member C (API, timeline eval), Member D (demo) | Drive / GitHub Release `model-v1`, **never committed** |
| `results/test_metrics.json` | Member D (release), report, slides | `docs/model/test_metrics.json` |
| `results/model_selection.json` | Report | numbers into `docs/model/experiments.md` |
| `runs/test_eval/*.png` | Report, slides | `docs/model/figures/` |
| `results/*_detections.jsonl` | Member C (timeline) | input to `src/timeline/builder.py` |

Only write numbers that came out of this notebook.